## Encoding and Transformer

### Encoder MLP
From observation data ( which includes observations and unseen data ),  
We put the data $x$ into the encoder $q_{\phi}(y \mid o )$, then sample "latent data vector" $y^{(i)} \sim q_{\phi}(y \mid o )$ .

Where, we model the distribution family of encoder as Normal distribution, whose mean is MLP

$$
    y^{(i)} \sim q_{\phi}(y \mid o ) = \mathcal{N}(z; \mu_\phi(x) , \Sigma_q),\quad \text{Where, } \Sigma_q = 0.1^2 I
$$

The following MLP is to make encoder mean MLP and then sample the latent vector $y^{(i)}$

In [1]:
import torch
from torch import nn

from src import make_ou_data
from src import ObsEncoder

data = make_ou_data(32, 40, 3)

obs = data["observation"]
obs_mask = data["obs_mask"]
times = data["times"]

"""
    ObsEncoder is the encoder MLP implementing, MLP mean and reparametrized sampling.
"""
encoder = ObsEncoder(obs_dim= obs.shape[-1], embed_dim=8)

"""
    Input : observation data & mask  (32, 40, 3) + (32, 40, 3)

    Output : sampled latent vector : (32, 40, 8)
"""

sampled_latent_obs = encoder(obs, obs_mask)

B,T,D =sampled_latent_obs.shape # (32, 40, 8)

print("obs:", obs.shape)
print("obs_mask:", obs_mask.shape, obs_mask.dtype)
print("y: " , sampled_latent_obs.shape)


obs: torch.Size([32, 40, 3])
obs_mask: torch.Size([32, 40]) torch.bool
y:  torch.Size([32, 40, 8])


## Transformer 

### Self-Attention mechanism

This cell is for explaining the basic attention mechanism for studying purpose.
You can ignore this cell, and directly go to the Positional Encoding block

In [2]:
import math

"""
    Make Linear Transform function ( y = W_y(.) ) for Q, K, V
"""
y = sampled_latent_obs

q_proj = nn.Linear(D, D, bias=False).to(
    device = y.device, dtype=y.dtype
)

k_proj = nn.Linear(D,D, bias=False).to(
    device = y.device, dtype=y.dtype
)

v_proj = nn.Linear(D,D, bias=False).to(
    device = y.device, dtype=y.dtype
)

"""
    make vector Q = y * W_x =  (D) * (D,D)  for all (B,T)

    Therefore, make the same y vector -> Q, K, V  vectors with own "parameter".
"""
Q = q_proj(y)
K = k_proj(y)
V = v_proj(y)

d_k = Q.shape[-1]

"""
    Q = (B,T,D) => (B) * [ [Q]_1^T , [Q]_2^T, ... , [Q]_D^T ]
    K^T = (B,D,T) => (B) * [ [K]_1 , [K]_2, ... , [K]_D ]

    ( Q @ K^T )_{t,T} = (B) * < Q_t | K_T >

    Therefore, scores : (B, T , T),  at each batch : <Q_t | K_T >/sqrt(d_k)

"""
scores = (
    Q @ K.transpose(-2,-1)
) / math.sqrt(d_k) 

"""
    using softmax to assign "exponential weight" according to normed inner product.
"""
weights = torch.softmax(scores, dim=-1)  # (B, T, T)

"""
    H = softmax( <Q | K>/sqrt(d_k)) V
"""
attention_output = weights @ V  # (B, T, D)



### Masked Self-Attention mechanism

In [ ]:

if obs_mask.dtype != torch.bool:
    raise TypeError("obs_mask must be a boolean tensor.")

if obs_mask.shape != y.shape[:2]:
    raise ValueError("obs_mask must have shape (B, T).")


"""
    tensor.any(dim=-1) : inspect if any "True" exists along with the last dim ( here, (B,T) -> T )  ( True>=1 : True /  True=0 : False)
    => for each batch, the following means :  only the first two batch have more than 1 True.

    [ [True], 
      [True],
      [False],
      ...,
      [False] ]

      
    tensor.all().item()  :  .all ( is every elemnt "True"? )  .item ( extract value from '0d' scalar tensor -> to value. )
    : if the tensor contains all True, then return True.

    => obs_mask.any(dim=-1).all().item()  :  check each batch of mask. then if "all batch" has at least one True, then return True.
"""
if not obs_mask.any(dim=-1).all().item():
    raise ValueError(
        "Each sequence must contain at least one observed key."
    )


"""
    reverse the obs_mask. for masked attention ( masked_fill method ).
    masked_fill method replaces the element according to the True position in the mask.
"""
key_padding_mask = ~obs_mask  # (B, T)

"""
    add an extra dim to match the tensor dimension for masking.
"""
blocked_keys = key_padding_mask.unsqueeze(1)  # (B, 1, T)


"""
    we need to replace "unseen scored" = related to unseen y-vector.

    we apply softmax function to the score. therefore, scores for unseed y-vector should be -inf.

    exp(-inf) = 0
    
"""
masked_scores = scores.masked_fill(
    blocked_keys,
    float("-inf"),
)  # (B, T, T)


masked_weights = torch.softmax(
    masked_scores,
    dim=-1,
)  # (B, T, T)


masked_attention_output = masked_weights @ V  # (B, T, D)

### Positional Encoding

positional encoding makes the vector of the same dimension as y-vector.

after appling self-attention to y (latent vector) , it still does note contain
time correlation structure because attention mixes all components.

Therefore, This PE must be included to properly express time correlation of series.

You can check the implementation PE in src/PE_Transformer.py

If the dimension of the latent vector $y, \text{dim: }D = 8 $,

$$
    2k = \{0,2,4,6\} \\
    2k+1 = \{1,3,5,7\}
$$


$$
PE(t) = 
\begin{pmatrix}
    \sin t \\ \cos t \\ \sin(0.1t) \\ \cos(0.1t) \\ \sin(0.01t) \\ \cos(0.01t) \\ \sin(0.001t) \\ \cos(0.001t)
\end{pmatrix}
$$

Where,
$$
    PE(t,2k) = \sin \left( \frac{t}{10000^{2k/D}} \right) \\
    PE(t,2k+1) = \cos \left( \frac{t}{10000^{2k/D}}\right)
$$

#### stacking tensor -> reshape

pe.reshape(B,T,D) : (B,T,D/2,2) => (B,T,D)

make (D/2,2) tensor rearrange to make (D)

$$
\begin{align*}
    &Stacked(t) = 
        \begin{pmatrix}
            \sin t & \cos t \\ \sin(0.1t) & \cos(0.1t) \\ \sin(0.01t) & \cos(0.01t) \\ \sin(0.001t) & \cos(0.001t)
        \end{pmatrix}, \quad \text{type :} (B,T,D/2,2)\\[1em]

    &Reshaped(t)=
        \begin{pmatrix}
            \sin t \\ \cos t \\ \sin(0.1t) \\ \cos(0.1t) \\ \sin(0.01t) \\ \cos(0.01t) \\ \sin(0.001t) \\ \cos(0.001t)
        \end{pmatrix}, \quad \text{type: } (B,T,D)
\end{align*} 
$$




In [ ]:

import math
import torch

from src import ACSSMTransformer, masking_unseen_data_Full_Assim


### Connect Transformer Encoder Layer

In [5]:
B,T,D = sampled_latent_obs.shape
transformer = ACSSMTransformer(D,2)

z_raw = transformer(sampled_latent_obs, times, obs_mask)

  

time_embedding: torch.Size([32, 40, 8])
transformer_input: torch.Size([32, 40, 8])
z_raw: torch.Size([32, 40, 8])


### Fill Unseen data using the closest history

Although using mask to conceal the key vector of the unseen data in the transformer, the result of transformer still contains the data from query vector of unseen data.

Therefore, we need to mask the z vector from unseed query vector, then replace them with the latest history value.

In [7]:

z = masking_unseen_data_Full_Assim(z_raw, obs_mask)

print("transformer_output: ", z_raw.shape)
print("z: ", z.shape)

transformer_output:  torch.Size([32, 40, 8])
z:  torch.Size([32, 40, 8])
